In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


In [2]:
# --- Paths (match your project structure) ---
DATASET_CLEAN = Path("../data/processed/dataset_clean.csv")
MACRO_SUMMARY = Path("../data/processed/output/macro_summary.csv")
TOPIC_SUMMARY = Path("../data/processed/output/topic_summary_with_macros.csv") 
DATASET_WITH_CLUSTERS = Path("../data/processed/output/dataset_with_clusters_and_macros.csv")

for p in [DATASET_CLEAN, MACRO_SUMMARY, TOPIC_SUMMARY, DATASET_WITH_CLUSTERS]:
    assert p.exists(), f"Missing: {p}"

DATASET_CLEAN, MACRO_SUMMARY, TOPIC_SUMMARY, DATASET_WITH_CLUSTERS


(PosixPath('../data/processed/dataset_clean.csv'),
 PosixPath('../data/processed/output/macro_summary.csv'),
 PosixPath('../data/processed/output/topic_summary_with_macros.csv'),
 PosixPath('../data/processed/output/dataset_with_clusters_and_macros.csv'))

In [3]:
# --- Load data ---
df_clean = pd.read_csv(DATASET_CLEAN, encoding="utf-8", encoding_errors="replace")
macro_summary = pd.read_csv(MACRO_SUMMARY, encoding="utf-8", encoding_errors="replace")
topic_summary = pd.read_csv(TOPIC_SUMMARY, encoding="utf-8", encoding_errors="replace")
df_all = pd.read_csv(DATASET_WITH_CLUSTERS, encoding="utf-8", encoding_errors="replace")

print("df_clean shape:", df_clean.shape)
print("macro_summary shape:", macro_summary.shape)
print("topic_summary shape:", topic_summary.shape)
print("df_all shape:", df_all.shape)

display(macro_summary.head())
display(topic_summary.head())


df_clean shape: (3531, 16)
macro_summary shape: (10, 5)
topic_summary shape: (30, 13)
df_all shape: (3531, 22)


,MacroId,MacroName,Count,MacroCategory,MacroSlug
0,3,Macro 3: meshes • direct volume rendering • ra...,1372,Meshes & Volume Rendering,meshes-volume-rendering
1,4,Macro 4: flow field • flow fields • vortices •...,634,Flow Fields & CFD Visualization,flow-fields-cfd
2,2,Macro 2: multidimensional scaling • dimensiona...,435,Dimensionality Reduction & Multivariate Plots,dimensionality-reduction-multivariate
3,1,Macro 1: surgical planning • virtual colonosco...,381,Surgical Planning & Tomography,surgical-planning-tomography
4,7,Macro 7: diffusion tensor • magnetic resonance...,258,Diffusion MRI & Tractography,diffusion-mri-tractography


,Topic,Count,Name,Representation,Representative_Docs,Category,MacroId,MacroName,CategoryShort,CategorySlug,CategoryFileSlug,MacroCategory,MacroSlug
0,0,555,0_mesh simplification_triangulation_marching c...,"['mesh simplification', 'triangulation', 'marc...",['Progressive Compression of Arbitrary Triangu...,mesh simplification • triangulation • marching...,3,Macro 3: meshes • direct volume rendering • ra...,Mesh Simplification & Surface Reconstruction,mesh-simplification-surface-reconstruction,t00-mesh-simplification,Meshes & Volume Rendering,meshes-volume-rendering
1,1,428,1_direct volume rendering_volume ray casting_v...,"['direct volume rendering', 'volume ray castin...",['Volume Ray Casting with Peak finding and Dif...,direct volume rendering • volume ray casting •...,3,Macro 3: meshes • direct volume rendering • ra...,Direct Volume Rendering & Ray Casting,direct-volume-rendering-ray-casting,t01-direct-volume-rendering,Meshes & Volume Rendering,meshes-volume-rendering
2,2,276,2_flow field_flow fields_vortices_computationa...,"['flow field', 'flow fields', 'vortices', 'com...",['Particle and texture based spatiotemporal vi...,flow field • flow fields • vortices • computat...,4,Macro 4: flow field • flow fields • vortices •...,Flow Fields & Vortices (CFD),flow-fields-vortices-cfd,t02-flow-fields-vortices,Flow Fields & CFD Visualization,flow-fields-cfd
3,3,183,3_multidimensional scaling_dimensionality redu...,"['multidimensional scaling', 'dimensionality r...",['Interactive Design and Visualization of Bran...,multidimensional scaling • dimensionality redu...,2,Macro 2: multidimensional scaling • dimensiona...,Dimensionality Reduction & Scatterplots,dimensionality-reduction-scatterplots,t03-dimensionality-reduction,Dimensionality Reduction & Multivariate Plots,dimensionality-reduction-multivariate
4,4,177,4_surgical planning_virtual colonoscopy_surgic...,"['surgical planning', 'virtual colonoscopy', '...",['Anatomy-based facial tissue modeling using t...,surgical planning • virtual colonoscopy • surg...,1,Macro 1: surgical planning • virtual colonosco...,Surgical Planning & Tomography,surgical-planning-tomography,t04-surgical-planning,Surgical Planning & Tomography,surgical-planning-tomography


In [4]:
# --- Macro short names + slugs (based on your MacroId/MacroName list) ---
MACRO_WEB = {
    0: {"MacroCategory": "Twitter & Social Media", "MacroSlug": "twitter-social-media"},
    1: {"MacroCategory": "Surgical Planning & Tomography", "MacroSlug": "surgical-planning-tomography"},
    2: {"MacroCategory": "Dimensionality Reduction & Multivariate Plots", "MacroSlug": "dimensionality-reduction-multivariate"},
    3: {"MacroCategory": "Meshes & Volume Rendering", "MacroSlug": "meshes-volume-rendering"},
    4: {"MacroCategory": "Flow Fields & CFD Visualization", "MacroSlug": "flow-fields-cfd"},
    5: {"MacroCategory": "Dashboards & Infographics", "MacroSlug": "dashboards-infographics"},
    6: {"MacroCategory": "VIS Literature & Bibliometrics", "MacroSlug": "vis-literature-bibliometrics"},
    7: {"MacroCategory": "Diffusion MRI & Tractography", "MacroSlug": "diffusion-mri-tractography"},
    8: {"MacroCategory": "Time Series & Temporal Patterns", "MacroSlug": "time-series-temporal-patterns"},
    9: {"MacroCategory": "Ultrasound Volume Rendering & Segmentation", "MacroSlug": "ultrasound-volume-rendering-segmentation"},
}

missing = sorted(set(range(10)) - set(MACRO_WEB.keys()))
assert not missing, f"Missing macro ids in mapping: {missing}"

MACRO_WEB


{0: {'MacroCategory': 'Twitter & Social Media',
  'MacroSlug': 'twitter-social-media'},
 1: {'MacroCategory': 'Surgical Planning & Tomography',
  'MacroSlug': 'surgical-planning-tomography'},
 2: {'MacroCategory': 'Dimensionality Reduction & Multivariate Plots',
  'MacroSlug': 'dimensionality-reduction-multivariate'},
 3: {'MacroCategory': 'Meshes & Volume Rendering',
  'MacroSlug': 'meshes-volume-rendering'},
 4: {'MacroCategory': 'Flow Fields & CFD Visualization',
  'MacroSlug': 'flow-fields-cfd'},
 5: {'MacroCategory': 'Dashboards & Infographics',
  'MacroSlug': 'dashboards-infographics'},
 6: {'MacroCategory': 'VIS Literature & Bibliometrics',
  'MacroSlug': 'vis-literature-bibliometrics'},
 7: {'MacroCategory': 'Diffusion MRI & Tractography',
  'MacroSlug': 'diffusion-mri-tractography'},
 8: {'MacroCategory': 'Time Series & Temporal Patterns',
  'MacroSlug': 'time-series-temporal-patterns'},
 9: {'MacroCategory': 'Ultrasound Volume Rendering & Segmentation',
  'MacroSlug': 'ultras

In [5]:
# --- Topic/Category short names + slugs (based on your CATEGORY table) ---
# Keys are Topic ids (0..29). Slugs are safe for URLs; CategoryFileSlug is safe for filenames.
TOPIC_WEB = {
    0: {"CategoryShort": "Mesh Simplification & Surface Reconstruction", "CategorySlug": "mesh-simplification-surface-reconstruction", "CategoryFileSlug": "t00-mesh-simplification"},
    1: {"CategoryShort": "Direct Volume Rendering & Ray Casting", "CategorySlug": "direct-volume-rendering-ray-casting", "CategoryFileSlug": "t01-direct-volume-rendering"},
    2: {"CategoryShort": "Flow Fields & Vortices (CFD)", "CategorySlug": "flow-fields-vortices-cfd", "CategoryFileSlug": "t02-flow-fields-vortices"},
    3: {"CategoryShort": "Dimensionality Reduction & Scatterplots", "CategorySlug": "dimensionality-reduction-scatterplots", "CategoryFileSlug": "t03-dimensionality-reduction"},
    4: {"CategoryShort": "Surgical Planning & Tomography", "CategorySlug": "surgical-planning-tomography", "CategoryFileSlug": "t04-surgical-planning"},
    5: {"CategoryShort": "Multi-Projector & High-Res Displays", "CategorySlug": "multi-projector-highres-displays", "CategoryFileSlug": "t05-multi-projector-displays"},
    6: {"CategoryShort": "Computational Topology & Contour Trees", "CategorySlug": "computational-topology-contour-trees", "CategoryFileSlug": "t06-contour-trees-topology"},
    7: {"CategoryShort": "Molecular Volume Rendering", "CategorySlug": "molecular-volume-rendering", "CategoryFileSlug": "t07-molecular-volume-rendering"},
    8: {"CategoryShort": "Diffusion MRI & Tractography", "CategorySlug": "diffusion-mri-tractography", "CategoryFileSlug": "t08-diffusion-mri"},
    9: {"CategoryShort": "Weather Forecast Visualization", "CategorySlug": "weather-forecast-visualization", "CategoryFileSlug": "t09-weather-forecast"},
    10: {"CategoryShort": "Sensemaking & Provenance", "CategorySlug": "sensemaking-provenance", "CategoryFileSlug": "t10-sensemaking-provenance"},
    11: {"CategoryShort": "Twitter & Social Media", "CategorySlug": "twitter-social-media", "CategoryFileSlug": "t11-twitter-social"},
    12: {"CategoryShort": "Unstructured Tetrahedral Volume Rendering", "CategorySlug": "unstructured-tetrahedral-volume-rendering", "CategoryFileSlug": "t12-tetrahedral-volume-rendering"},
    13: {"CategoryShort": "Sports Pattern Mining", "CategorySlug": "sports-pattern-mining", "CategoryFileSlug": "t13-sports-pattern-mining"},
    14: {"CategoryShort": "NLP & Question Answering", "CategorySlug": "nlp-question-answering", "CategoryFileSlug": "t14-nlp-question-answering"},
    15: {"CategoryShort": "IEEE VIS Bibliometrics", "CategorySlug": "ieee-vis-bibliometrics", "CategoryFileSlug": "t15-ieee-vis-bibliometrics"},
    16: {"CategoryShort": "Treemaps & Tree Layouts", "CategorySlug": "treemaps-tree-layouts", "CategoryFileSlug": "t16-treemaps"},
    17: {"CategoryShort": "Immersive Analytics (AR/VR)", "CategorySlug": "immersive-analytics-ar-vr", "CategoryFileSlug": "t17-immersive-analytics"},
    18: {"CategoryShort": "Dashboards & Infographics", "CategorySlug": "dashboards-infographics", "CategoryFileSlug": "t18-dashboards-infographics"},
    19: {"CategoryShort": "Colormaps & Color Perception", "CategorySlug": "colormaps-color-perception", "CategoryFileSlug": "t19-colormaps"},
    20: {"CategoryShort": "Eye Tracking & Gaze Analysis", "CategorySlug": "eye-tracking-gaze-analysis", "CategoryFileSlug": "t20-eye-tracking"},
    21: {"CategoryShort": "Tensor Field Topology", "CategorySlug": "tensor-field-topology", "CategoryFileSlug": "t21-tensor-fields"},
    22: {"CategoryShort": "Line Charts & Temporal Patterns", "CategorySlug": "line-charts-temporal-patterns", "CategoryFileSlug": "t22-line-charts"},
    23: {"CategoryShort": "Gene Expression & Genomics", "CategorySlug": "gene-expression-genomics", "CategoryFileSlug": "t23-gene-expression"},
    24: {"CategoryShort": "Visualization Design & Perception", "CategorySlug": "visualization-design-perception", "CategoryFileSlug": "t24-viz-design"},
    25: {"CategoryShort": "Narrative Visualization & Storytelling", "CategorySlug": "narrative-visualization-storytelling", "CategoryFileSlug": "t25-narrative-viz"},
    26: {"CategoryShort": "Cell Imaging & Machine Learning", "CategorySlug": "cell-imaging-machine-learning", "CategoryFileSlug": "t26-cell-imaging-ml"},
    27: {"CategoryShort": "Financial Monitoring & Anomaly Detection", "CategorySlug": "financial-monitoring-anomaly-detection", "CategoryFileSlug": "t27-financial-monitoring"},
    28: {"CategoryShort": "Illustrative Volume Rendering", "CategorySlug": "illustrative-volume-rendering", "CategoryFileSlug": "t28-illustrative-rendering"},
    29: {"CategoryShort": "Ultrasound Volume Rendering", "CategorySlug": "ultrasound-volume-rendering", "CategoryFileSlug": "t29-ultrasound"},
}

# If TOPIC_SUMMARY contains a Topic column, sanity-check coverage:
if "Topic" in topic_summary.columns:
    topics = sorted(pd.unique(topic_summary["Topic"].dropna().astype(int)))
    missing_topics = [t for t in topics if t not in TOPIC_WEB]
    if missing_topics:
        print("WARNING: Unmapped Topic id(s):", missing_topics)
    else:
        print("All topic ids in topic_summary are mapped:", len(topics))

TOPIC_WEB


All topic ids in topic_summary are mapped: 30


{0: {'CategoryShort': 'Mesh Simplification & Surface Reconstruction',
  'CategorySlug': 'mesh-simplification-surface-reconstruction',
  'CategoryFileSlug': 't00-mesh-simplification'},
 1: {'CategoryShort': 'Direct Volume Rendering & Ray Casting',
  'CategorySlug': 'direct-volume-rendering-ray-casting',
  'CategoryFileSlug': 't01-direct-volume-rendering'},
 2: {'CategoryShort': 'Flow Fields & Vortices (CFD)',
  'CategorySlug': 'flow-fields-vortices-cfd',
  'CategoryFileSlug': 't02-flow-fields-vortices'},
 3: {'CategoryShort': 'Dimensionality Reduction & Scatterplots',
  'CategorySlug': 'dimensionality-reduction-scatterplots',
  'CategoryFileSlug': 't03-dimensionality-reduction'},
 4: {'CategoryShort': 'Surgical Planning & Tomography',
  'CategorySlug': 'surgical-planning-tomography',
  'CategoryFileSlug': 't04-surgical-planning'},
 5: {'CategoryShort': 'Multi-Projector & High-Res Displays',
  'CategorySlug': 'multi-projector-highres-displays',
  'CategoryFileSlug': 't05-multi-projector-

In [6]:
# --- Enrich + save macro_summary ---
macro_summary["MacroId"] = macro_summary["MacroId"].astype(int)

macro_summary["MacroCategory"] = macro_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroCategory"))
macro_summary["MacroSlug"] = macro_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroSlug"))

unmapped = macro_summary[macro_summary["MacroCategory"].isna()]["MacroId"].unique().tolist()
if unmapped:
    print("WARNING: Unmapped MacroId(s) in macro_summary:", unmapped)

macro_summary.to_csv(MACRO_SUMMARY, index=False)
print("Updated file written:", MACRO_SUMMARY.resolve())

display(macro_summary.head(20))


Updated file written: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/output/macro_summary.csv


,MacroId,MacroName,Count,MacroCategory,MacroSlug
0,3,Macro 3: meshes • direct volume rendering • ra...,1372,Meshes & Volume Rendering,meshes-volume-rendering
1,4,Macro 4: flow field • flow fields • vortices •...,634,Flow Fields & CFD Visualization,flow-fields-cfd
2,2,Macro 2: multidimensional scaling • dimensiona...,435,Dimensionality Reduction & Multivariate Plots,dimensionality-reduction-multivariate
3,1,Macro 1: surgical planning • virtual colonosco...,381,Surgical Planning & Tomography,surgical-planning-tomography
4,7,Macro 7: diffusion tensor • magnetic resonance...,258,Diffusion MRI & Tractography,diffusion-mri-tractography
5,6,Macro 6: visualisations • ieee vis • scientifi...,176,VIS Literature & Bibliometrics,vis-literature-bibliometrics
6,0,Macro 0: twitter • social media • tweets • media,104,Twitter & Social Media,twitter-social-media
7,5,Macro 5: dashboards • infographics • dashboard...,99,Dashboards & Infographics,dashboards-infographics
8,8,Macro 8: line charts • multidimensional scalin...,44,Time Series & Temporal Patterns,time-series-temporal-patterns
9,9,Macro 9: direct volume rendering • ultrasound ...,28,Ultrasound Volume Rendering & Segmentation,ultrasound-volume-rendering-segmentation


In [7]:
# --- Enrich + save topic_summary ---
if "Topic" in topic_summary.columns:
    topic_summary["Topic"] = topic_summary["Topic"].astype(int)
    topic_summary["CategoryShort"] = topic_summary["Topic"].map(lambda t: TOPIC_WEB.get(t, {}).get("CategoryShort"))
    topic_summary["CategorySlug"] = topic_summary["Topic"].map(lambda t: TOPIC_WEB.get(t, {}).get("CategorySlug"))
    topic_summary["CategoryFileSlug"] = topic_summary["Topic"].map(lambda t: TOPIC_WEB.get(t, {}).get("CategoryFileSlug"))

    # Also bring MacroCategory/MacroSlug onto the topic table, if MacroId exists there
    if "MacroId" in topic_summary.columns:
        topic_summary["MacroId"] = topic_summary["MacroId"].astype(int)
        topic_summary["MacroCategory"] = topic_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroCategory"))
        topic_summary["MacroSlug"] = topic_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroSlug"))

    unmapped_topics = topic_summary[topic_summary["CategoryShort"].isna()]["Topic"].unique().tolist()
    if unmapped_topics:
        print("WARNING: Unmapped Topic(s) in topic_summary:", unmapped_topics)

    topic_summary.to_csv(TOPIC_SUMMARY, index=False)
    print("Updated file written:", TOPIC_SUMMARY.resolve())

display(topic_summary.head(30))


Updated file written: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/output/topic_summary_with_macros.csv


,Topic,Count,Name,Representation,Representative_Docs,Category,MacroId,MacroName,CategoryShort,CategorySlug,CategoryFileSlug,MacroCategory,MacroSlug
0,0,555,0_mesh simplification_triangulation_marching c...,"['mesh simplification', 'triangulation', 'marc...",['Progressive Compression of Arbitrary Triangu...,mesh simplification • triangulation • marching...,3,Macro 3: meshes • direct volume rendering • ra...,Mesh Simplification & Surface Reconstruction,mesh-simplification-surface-reconstruction,t00-mesh-simplification,Meshes & Volume Rendering,meshes-volume-rendering
1,1,428,1_direct volume rendering_volume ray casting_v...,"['direct volume rendering', 'volume ray castin...",['Volume Ray Casting with Peak finding and Dif...,direct volume rendering • volume ray casting •...,3,Macro 3: meshes • direct volume rendering • ra...,Direct Volume Rendering & Ray Casting,direct-volume-rendering-ray-casting,t01-direct-volume-rendering,Meshes & Volume Rendering,meshes-volume-rendering
2,2,276,2_flow field_flow fields_vortices_computationa...,"['flow field', 'flow fields', 'vortices', 'com...",['Particle and texture based spatiotemporal vi...,flow field • flow fields • vortices • computat...,4,Macro 4: flow field • flow fields • vortices •...,Flow Fields & Vortices (CFD),flow-fields-vortices-cfd,t02-flow-fields-vortices,Flow Fields & CFD Visualization,flow-fields-cfd
3,3,183,3_multidimensional scaling_dimensionality redu...,"['multidimensional scaling', 'dimensionality r...",['Interactive Design and Visualization of Bran...,multidimensional scaling • dimensionality redu...,2,Macro 2: multidimensional scaling • dimensiona...,Dimensionality Reduction & Scatterplots,dimensionality-reduction-scatterplots,t03-dimensionality-reduction,Dimensionality Reduction & Multivariate Plots,dimensionality-reduction-multivariate
4,4,177,4_surgical planning_virtual colonoscopy_surgic...,"['surgical planning', 'virtual colonoscopy', '...",['Anatomy-based facial tissue modeling using t...,surgical planning • virtual colonoscopy • surg...,1,Macro 1: surgical planning • virtual colonosco...,Surgical Planning & Tomography,surgical-planning-tomography,t04-surgical-planning,Surgical Planning & Tomography,surgical-planning-tomography
5,5,159,5_projector_high resolution displays_computer ...,"['projector', 'high resolution displays', 'com...",['Scalable alignment of large-format multi-pro...,projector • high resolution displays • compute...,1,Macro 1: surgical planning • virtual colonosco...,Multi-Projector & High-Res Displays,multi-projector-highres-displays,t05-multi-projector-displays,Surgical Planning & Tomography,surgical-planning-tomography
6,6,153,6_computational topology_contour trees_topolog...,"['computational topology', 'contour trees', 't...",['Efficient Computation of Morse-Smale Complex...,computational topology • contour trees • topol...,2,Macro 2: multidimensional scaling • dimensiona...,Computational Topology & Contour Trees,computational-topology-contour-trees,t06-contour-trees-topology,Dimensionality Reduction & Multivariate Plots,dimensionality-reduction-multivariate
7,7,146,7_molecular_molecules_volume rendering algorit...,"['molecular', 'molecules', 'volume rendering a...",['Characterizing Molecular Interactions in Che...,molecular • molecules • volume rendering algor...,4,Macro 4: flow field • flow fields • vortices •...,Molecular Volume Rendering,molecular-volume-rendering,t07-molecular-volume-rendering,Flow Fields & CFD Visualization,flow-fields-cfd
8,8,136,8_diffusion tensor_magnetic resonance imaging_...,"['diffusion tensor', 'magnetic resonance imagi...",['Glyph-Based Comparative Visualization for Di...,diffusion tensor • magnetic resonance imaging ...,7,Macro 7: diffusion tensor • magnetic resonance...,Diffusion MRI & Tractography,diffusion-mri-tractography,t08-diffusion-mri,Diffusion MRI & Tractography,diffusion-mri-tractography
9,9,126,9_weather forecasting_weather prediction_weath...,"['weather forecasting', 'weather prediction', ...",['

In [8]:
# --- (Optional) write MacroCategory + Cluster (id) + ClusterShort (name) into dataset_clean.csv ---
# NOTE: In your pipeline, `topic_summary_with_macros.csv` uses column `Topic`,
# but `dataset_with_clusters_and_macros.csv` typically stores the same id in column `Cluster`.
# This block supports BOTH: it will use `Topic` if present, otherwise `Cluster`.

MERGE_KEY = "DOI"  # change if your join key is different

# Pick the column that contains the topic/cluster id
cluster_col = "Topic" if "Topic" in df_all.columns else ("Cluster" if "Cluster" in df_all.columns else None)
if cluster_col is None:
    raise ValueError("dataset_with_clusters_and_macros.csv is missing both 'Topic' and 'Cluster' columns.")

need_cols = [MERGE_KEY, "MacroId", cluster_col]
missing_cols = [c for c in need_cols if c not in df_all.columns]
if missing_cols:
    raise ValueError(f"dataset_with_clusters_and_macros.csv is missing columns: {missing_cols}")

# Build map + normalize name to `Cluster`
df_map = df_all[need_cols].copy().rename(columns={cluster_col: "Cluster"})

# Avoid exploding rows if there are duplicates in the merge key
if df_map[MERGE_KEY].duplicated().any():
    df_map = df_map.drop_duplicates(subset=[MERGE_KEY], keep="first")

df_out = df_clean.merge(df_map, on=MERGE_KEY, how="left")

# Macro labels
df_out["MacroCategory"] = df_out["MacroId"].map(
    lambda m: MACRO_WEB.get(int(m), {}).get("MacroCategory") if pd.notna(m) else np.nan
)
df_out["MacroSlug"] = df_out["MacroId"].map(
    lambda m: MACRO_WEB.get(int(m), {}).get("MacroSlug") if pd.notna(m) else np.nan
)

# Cluster/topic labels (short name)
df_out["ClusterShort"] = df_out["Cluster"].map(
    lambda t: TOPIC_WEB.get(int(t), {}).get("CategoryShort") if pd.notna(t) else np.nan
)
df_out["ClusterSlug"] = df_out["Cluster"].map(
    lambda t: TOPIC_WEB.get(int(t), {}).get("CategorySlug") if pd.notna(t) else np.nan
)
df_out["ClusterFileSlug"] = df_out["Cluster"].map(
    lambda t: TOPIC_WEB.get(int(t), {}).get("CategoryFileSlug") if pd.notna(t) else np.nan
)

# Report missing mappings
print("Rows with missing MacroCategory:", int(df_out["MacroCategory"].isna().sum()), "/", len(df_out))
print("Rows with missing ClusterShort:", int(df_out["ClusterShort"].isna().sum()), "/", len(df_out))

# Save
df_out.to_csv(DATASET_CLEAN, index=False)
print("Updated file written:", DATASET_CLEAN.resolve())

display(df_out.head())


Rows with missing MacroCategory: 0 / 3531
Rows with missing ClusterShort: 0 / 3531
Updated file written: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/dataset_clean.csv


,Conference,Year,Title,DOI,PaperType,Abstract,AuthorNames-Deduped,AuthorAffiliation,InternalReferences,AuthorKeywords,...,Downloads_Xplore,Award,GraphicsReplicabilityStamp,MacroId,Cluster,MacroCategory,MacroSlug,ClusterShort,ClusterSlug,ClusterFileSlug
0,Vis,2024,Interactive Design-of-Experiments: Optimizing ...,10.1109/tvcg.2024.3456356,J,The optimization of cooling systems is importa...,Rainer Splechtna;Majid Behravan;Mario Jelovic;...,"VRVis Research Center in Vienna, Austria;Virgi...",10.1109/tvcg.2013.124;10.1109/tvcg.2008.145;10...,Parameter space exploration,...,234.0,NaN,NaN,3,0,Meshes & Volume Rendering,meshes-volume-rendering,Mesh Simplification & Surface Reconstruction,mesh-simplification-surface-reconstruction,t00-mesh-simplification
1,Vis,2024,Towards Dataset-Scale and Feature-Oriented Eva...,10.1109/tvcg.2024.3456398,J,Recent advancements in Large Language Models (...,Sam Yu-Te Lee;Aryaman Bahukhandi;Dongyu Liu;Kw...,"University of California, USA;University of Ca...",10.1109/tvcg.2017.2743858;10.1109/tvcg.2017.27...,"Visual analytics,prompt engineering,,,text sum...",...,386.0,NaN,NaN,7,14,Diffusion MRI & Tractography,diffusion-mri-tractography,NLP & Question Answering,nlp-question-answering,t14-nlp-question-answering
2,Vis,2024,KNowNEt:Guided Health Information Seeking from...,10.1109/tvcg.2024.3456364,J,The increasing reliance on Large Language Mode...,Youfu Yan;Yu Hou;Yongkang Xiao;Rui Zhang;Qianw...,Department of Computer Science and Engineering...,10.1109/tvcg.2022.3209408;10.1109/tvcg.2023.33...,"Human-AI interactions,knowledge graph,,,conver...",...,632.0,HM,NaN,7,14,Diffusion MRI & Tractography,diffusion-mri-tractography,NLP & Question Answering,nlp-question-answering,t14-nlp-question-answering
3,Vis,2024,VisEval: A Benchmark for Data Visualization in...,10.1109/tvcg.2024.3456320,J,Translating natural language to visualization ...,Nan Chen;Yuge Zhang;Jiahang Xu;Kan Ren;Yuqing ...,"Microsoft Research, USA;Microsoft Research, US...",10.1109/infvis.2005.1532136;10.1109/tvcg.2015....,"Visualization evaluation,automatic visualizati...",...,625.0,BP,NaN,7,14,Diffusion MRI & Tractography,diffusion-mri-tractography,NLP & Question Answering,nlp-question-answering,t14-nlp-question-answering
4,Vis,2024,PUREsuggest: Citation-Based Literature Search ...,10.1109/tvcg.2024.3456199,J,Citations allow quickly identifying related re...,Fabian Beck 0001,"University of Bamberg, Germany",10.1109/tvcg.2015.2467757;10.1109/tvcg.2016.25...,"Scientific literature search,citation network ...",...,165.0,NaN,NaN,6,15,VIS Literature & Bibliometrics,vis-literature-bibliometrics,IEEE VIS Bibliometrics,ieee-vis-bibliometrics,t15-ieee-vis-bibliometrics
